In [ ]:
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from pathlib import Path
from statsmodels.regression.mixed_linear_model import MixedLM
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

In [ ]:
freq_list = ['freq_0.5','freq_0.75','freq_1.0']
attr_list = ['gain','velocity','speed']
axis_list = ['horizontal','vertical']
region_list = ['positive_half_cycle','negative_half_cycle','whole_cycle']

base_json_path = './'

In [ ]:
class FeatureSelectionPipeline:
    """
    Feature selection pipeline for analyzing 54 JSON files.
    Each file represents a feature combination:
    freq_{freq}_{attribute}_{axis}_{cycle_region}.json
    """

    def __init__(self, data_dir, verbose=True):
        """
        Args:
            data_dir: Directory containing the JSON files
            verbose: Print progress information
        """
        self.data_dir = Path(data_dir)
        self.verbose = verbose
        self.file_info = []
        self.features_data = {}
        self.results = {}

        # Parse file structure
        self.frequencies = ['0.5', '0.75', '1.0']
        self.attributes = ['gain', 'speed', 'velocity']
        self.axes = ['horizontal', 'vertical']
        self.cycle_regions = ['positive_half', 'negative_half', 'whole_cycle']

    def parse_filename(self, filename):
        """
        Parse filename to extract feature information.

        Example: freq_0.75_speed_horizontal_negative_half_cycle.json
        Returns: {
            'frequency': '0.75',
            'attribute': 'speed',
            'axis': 'horizontal',
            'cycle_region': 'negative_half_cycle'
        }
        """
        # Remove .json extension
        name = filename.replace('.json', '')

        # Split by underscore
        parts = name.split('_')

        # Parse: freq_{freq}_{attribute}_{axis}_{cycle_region}
        # freq_0.75_speed_horizontal_negative_half_cycle.json
        # [freq] [0.75] [speed] [horizontal] [negative] [half] [cycle]

        # Handle the cycle region which may have multiple parts
        if 'positive' in name:
            cycle_region = 'positive_half_cycle'
        elif 'negative' in name:
            cycle_region = 'negative_half_cycle'
        else:
            cycle_region = 'whole_cycle'

        # Extract frequency (second part)
        frequency = parts[1] if len(parts) > 1 else None

        # Extract attribute (third part)
        attribute = parts[2] if len(parts) > 2 else None

        # Extract axis (fourth part)
        axis = parts[3] if len(parts) > 3 else None

        return {
            'filename': filename,
            'frequency': frequency,
            'attribute': attribute,
            'axis': axis,
            'cycle_region': cycle_region,
            'feature_name': f"{attribute}_{axis}_{cycle_region}_freq{frequency}"
        }

    def load_all_files(self):
        """
        Load all JSON files from the data directory.
        """
        print("\n" + "="*70)
        print("LOADING DATA FILES")
        print("="*70)

        json_files = list(self.data_dir.glob('*.json'))
        print(f"Found {len(json_files)} JSON files")

        for file_path in sorted(json_files):
            file_info = self.parse_filename(file_path.name)

            if self.verbose:
                print(f"\nLoading: {file_path.name}")
                print(f"  Frequency: {file_info['frequency']}")
                print(f"  Attribute: {file_info['attribute']}")
                print(f"  Axis: {file_info['axis']}")
                print(f"  Cycle Region: {file_info['cycle_region']}")

            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)

                # Store the data
                file_info['data'] = data
                self.features_data[file_info['feature_name']] = data
                self.file_info.append(file_info)

                # Show data shape
                if isinstance(data, list) and len(data) > 0:
                    if isinstance(data[0], list) and len(data[0]) > 0:
                        n_subjects = len(data[0])
                        n_samples = len(data[0][0]) if n_subjects > 0 else 0
                        print(f"  ✓ Loaded: {len(data)} classes, {n_subjects} subjects, {n_samples} samples")

            except Exception as e:
                print(f"  ✗ Error loading {file_path.name}: {e}")

        print(f"\n✅ Successfully loaded {len(self.file_info)} files")

        # Create a summary DataFrame
        self.file_summary = pd.DataFrame(self.file_info)
        self.file_summary = self.file_summary.drop('data', axis=1, errors='ignore')

        print("\nFile Summary:")
        print(f"  Frequencies: {self.file_summary['frequency'].unique().tolist()}")
        print(f"  Attributes: {self.file_summary['attribute'].unique().tolist()}")
        print(f"  Axes: {self.file_summary['axis'].unique().tolist()}")
        print(f"  Cycle Regions: {self.file_summary['cycle_region'].unique().tolist()}")

        return self.file_info

    def prepare_data_for_lmm(self, feature_name, class_container):
        """
        Prepare data from class container for LMM analysis.

        Args:
            feature_name: Name of the feature
            class_container: Nested list structure [class][subject][samples]
        """
        rows = []
        class_names = ['A', 'B', 'C']

        for class_idx, class_data in enumerate(class_container):
            class_name = class_names[class_idx]

            for subject_idx, subject_samples in enumerate(class_data):
                subject_id = f"{class_name}_{subject_idx}"

                for t_idx, value in enumerate(subject_samples):
                    rows.append({
                        'value': value,
                        'class_label': class_name,
                        'subject_id': subject_id,
                        'time': t_idx,
                        'time_scaled': t_idx / (len(subject_samples) - 1) if len(subject_samples) > 1 else 0
                    })

        df = pd.DataFrame(rows)
        df['class_label'] = pd.Categorical(df['class_label'], categories=class_names)
        df['subject_id'] = pd.Categorical(df['subject_id'])

        return df

    def evaluate_feature_with_lmm(self, df, feature_name):
        """
        Evaluate a single feature using LMM.
        """
        try:
            # Fit LMM
            model = MixedLM.from_formula(
                'value ~ class_label * time_scaled',
                data=df,
                groups=df['subject_id'],
                re_formula='1 + time_scaled'
            )

            result = model.fit(method='lbfgs', maxiter=1000, reml=True)

            if not result.converged:
                # Try simpler random effects
                model = MixedLM.from_formula(
                    'value ~ class_label * time_scaled',
                    data=df,
                    groups=df['subject_id'],
                    re_formula='1'
                )
                result = model.fit(method='lbfgs', maxiter=1000, reml=True)

            # Extract metrics
            metrics = {
                'feature': feature_name,
                'converged': result.converged,
                'log_likelihood': result.llf,
                'aic': result.aic,
                'bic': result.bic,
                'class_b_pvalue': result.pvalues.get('class_label[T.B]', np.nan),
                'class_c_pvalue': result.pvalues.get('class_label[T.C]', np.nan),
                'time_pvalue': result.pvalues.get('time_scaled', np.nan),
                'class_b_time_pvalue': result.pvalues.get('class_label[T.B]:time_scaled', np.nan),
                'class_c_time_pvalue': result.pvalues.get('class_label[T.C]:time_scaled', np.nan),
                'class_b_coef': result.params.get('class_label[T.B]', 0),
                'class_c_coef': result.params.get('class_label[T.C]', 0),
                'time_coef': result.params.get('time_scaled', 0),
                'class_b_time_coef': result.params.get('class_label[T.B]:time_scaled', 0),
                'class_c_time_coef': result.params.get('class_label[T.C]:time_scaled', 0),
            }

            # Compute discriminative score (combine class differences)
            p_values = [metrics['class_b_pvalue'], metrics['class_c_pvalue']]
            valid_p = [p for p in p_values if not np.isnan(p) and p > 0]
            if valid_p:
                avg_p = np.mean(valid_p)
                metrics['discriminative_score'] = -np.log10(avg_p + 1e-10)
            else:
                metrics['discriminative_score'] = 0

            # Effect size (combined class differences)
            metrics['effect_size'] = abs(metrics['class_b_coef']) + abs(metrics['class_c_coef'])

            return metrics

        except Exception as e:
            if self.verbose:
                print(f"  LMM failed for {feature_name}: {str(e)[:100]}")
            return {'feature': feature_name, 'converged': False, 'discriminative_score': 0}

    def evaluate_all_features(self):
        """
        Evaluate all loaded features using LMM.
        """
        print("\n" + "="*70)
        print("EVALUATING ALL FEATURES WITH LMM")
        print("="*70)
        print(f"Total features: {len(self.features_data)}")
        print("="*70 + "\n")

        results = []

        for i, (feature_name, class_container) in enumerate(self.features_data.items()):
            if self.verbose:
                print(f"[{i+1}/{len(self.features_data)}] Evaluating: {feature_name}")

            # Prepare data
            df = self.prepare_data_for_lmm(feature_name, class_container)

            # Evaluate
            metrics = self.evaluate_feature_with_lmm(df, feature_name)

            # Parse the feature components
            parts = feature_name.split('_')
            metrics['attribute'] = parts[0] if len(parts) > 0 else 'unknown'
            metrics['axis'] = parts[1] if len(parts) > 1 else 'unknown'
            metrics['cycle_region'] = '_'.join(parts[2:-1]) if len(parts) > 3 else 'unknown'
            metrics['frequency'] = parts[-1].replace('freq', '') if parts[-1].startswith('freq') else 'unknown'

            results.append(metrics)

            if self.verbose and metrics['converged']:
                print(f"  ✓ Score: {metrics['discriminative_score']:.4f} | "
                      f"p-values: B={metrics['class_b_pvalue']:.4f}, C={metrics['class_c_pvalue']:.4f}")
            elif self.verbose:
                print(f"  ✗ Did not converge")

        # Store results
        self.results['lmm'] = pd.DataFrame(results)
        self.results['lmm'] = self.results['lmm'].sort_values('discriminative_score', ascending=False)

        print(f"\n{'='*70}")
        print("TOP FEATURES BY DISCRIMINATIVE SCORE")
        print(f"{'='*70}")
        display_cols = ['feature', 'attribute', 'axis', 'frequency', 'cycle_region',
                       'discriminative_score', 'class_b_pvalue', 'class_c_pvalue']
        print(self.results['lmm'][display_cols].head(10).to_string(index=False))

        return self.results['lmm']

    def extract_subject_features(self, class_container, feature_name):
        """
        Extract summary features for each subject.
        """
        features = []
        labels = []

        for class_idx, class_data in enumerate(class_container):
            for subject_data in class_data:
                # Compute summary statistics
                features.extend([
                    np.mean(subject_data),           # Mean
                    np.std(subject_data),            # Std
                    np.percentile(subject_data, 25), # 25th percentile
                    np.percentile(subject_data, 50), # Median
                    np.percentile(subject_data, 75), # 75th percentile
                    np.max(subject_data) - np.min(subject_data),  # Range
                    np.mean(np.diff(subject_data)),  # Trend
                    np.std(np.diff(subject_data)),   # Variability
                ])
                labels.append(class_idx)

        return np.array(features), np.array(labels)

    def random_forest_importance(self):
      """
      Use Random Forest to identify important features.
      """
      print("\n" + "="*70)
      print("FEATURE IMPORTANCE USING RANDOM FOREST")
      print("="*70)

      feature_names = list(self.features_data.keys())
      stat_names = ['mean', 'std', 'p25', 'p50', 'p75', 'range', 'trend', 'variability']

      # First, let's check what we're working with
      print(f"Number of features: {len(feature_names)}")
      print(f"Number of statistics per feature: {len(stat_names)}")
      print(f"Total feature dimensions: {len(feature_names) * len(stat_names)}")

      # First, let's understand the data structure
      print("\nChecking class sizes:")
      class_names = ['A', 'B', 'C']
      class_sizes = []

      for class_idx, class_name in enumerate(class_names):
        # Use the first feature to check class size
        first_container = self.features_data[feature_names[0]]
        n_subjects = len(first_container[class_idx])
        class_sizes.append(n_subjects)
        print(f"  Class {class_name}: {n_subjects} subjects")

      n_subjects_total = sum(class_sizes)
      print(f"Total subjects: {n_subjects_total}")

      # Initialize feature matrix
      n_features_total = len(feature_names) * len(stat_names)

      X = np.zeros((n_subjects_total, n_features_total))
      y = np.zeros(n_subjects_total)
      subject_ids = []

      class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
      class_weight_dict= {i:class_weights[i] for i in range(len(class_weights))}

      subject_idx = 0

      for class_idx, class_name in enumerate(['A', 'B', 'C']):
          n_subjects_per_class = len(self.features_data[feature_names[0]][class_idx])
          for subj_idx in range(n_subjects_per_class):
              feature_idx = 0

              for feature_name in feature_names:
                  class_container = self.features_data[feature_name]
                  subject_data = class_container[class_idx][subj_idx]

                  # Compute summary statistics
                  stats = [
                      np.mean(subject_data),
                      np.std(subject_data),
                      np.percentile(subject_data, 25),
                      np.percentile(subject_data, 50),
                      np.percentile(subject_data, 75),
                      np.max(subject_data) - np.min(subject_data),
                      np.mean(np.diff(subject_data)) if len(subject_data) > 1 else 0,
                      np.std(np.diff(subject_data)) if len(subject_data) > 1 else 0,
                  ]

                  # Assign to X matrix
                  for stat_val in stats:
                      X[subject_idx, feature_idx] = stat_val
                      feature_idx += 1

              y[subject_idx] = class_idx
              subject_ids.append(f"{class_name}_{subj_idx}")
              subject_idx += 1

      print(f"\nFeature matrix shape: {X.shape}")
      print(f"Labels shape: {y.shape}")

      # Scale features
      scaler = StandardScaler()
      X_scaled = scaler.fit_transform(X)

      # Random Forest
      rf = RandomForestClassifier(
          n_estimators=100,
          max_depth=10,
          random_state=42,
          n_jobs=-1,
          class_weight=class_weight_dict
      )

      # Cross-validation
      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
      cv_scores = cross_val_score(rf, X_scaled, y, cv=cv, scoring='accuracy')

      print(f"\nCross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

      # Fit on full data
      rf.fit(X_scaled, y)

      # Get feature importance (should match n_features_total)
      importance = rf.feature_importances_
      print(f"Importance length: {len(importance)}")

      # Create feature names
      feature_names_expanded = []
      for feature_name in feature_names:
          for stat in stat_names:
              feature_names_expanded.append(f"{feature_name}_{stat}")

      try:
        # Create DataFrame
        importance_df = pd.DataFrame({
            'feature': feature_names_expanded,
            'importance': importance
        }).sort_values('importance', ascending=False)
      except Exception as e:
        print(f'Caught {e}, feature_names_expanded and importance returned')
        return feature_names_expanded, importance

      # Aggregate by base feature
      importance_df['base_feature'] = importance_df['feature'].str.rsplit('_', n=1).str[0]
      agg_importance = importance_df.groupby('base_feature')['importance'].sum().sort_values(ascending=False)

      self.results['rf_importance'] = importance_df
      self.results['rf_agg_importance'] = agg_importance

      print(f"\nTop 10 features by importance:")
      for feature, importance_val in agg_importance.head(10).items():
          print(f"  {feature}: {importance_val:.4f}")

      # Also show top individual statistics
      print(f"\nTop 10 individual statistics:")
      for i, row in importance_df.head(10).iterrows():
          print(f"  {row['feature']}: {row['importance']:.4f}")

      return agg_importance

    def analyze_by_feature_groups(self):
        """
        Analyze which feature groups (attribute, axis, frequency, cycle_region)
        are most discriminative.
        """
        if 'lmm' not in self.results:
            print("Run evaluate_all_features() first")
            return

        print("\n" + "="*70)
        print("ANALYSIS BY FEATURE GROUPS")
        print("="*70)

        df = self.results['lmm']

        # By attribute
        print("\nBy Attribute:")
        attr_means = df.groupby('attribute')['discriminative_score'].agg(['mean', 'std', 'count'])
        print(attr_means.sort_values('mean', ascending=False))

        # By axis
        print("\nBy Axis:")
        axis_means = df.groupby('axis')['discriminative_score'].agg(['mean', 'std', 'count'])
        print(axis_means.sort_values('mean', ascending=False))

        # By frequency
        print("\nBy Frequency:")
        freq_means = df.groupby('frequency')['discriminative_score'].agg(['mean', 'std', 'count'])
        print(freq_means.sort_values('mean', ascending=False))

        # By cycle region
        print("\nBy Cycle Region:")
        cycle_means = df.groupby('cycle_region')['discriminative_score'].agg(['mean', 'std', 'count'])
        print(cycle_means.sort_values('mean', ascending=False))

        # Store group results
        self.results['group_analysis'] = {
            'attribute': attr_means,
            'axis': axis_means,
            'frequency': freq_means,
            'cycle_region': cycle_means
        }

        # Find best combination
        print("\n" + "="*70)
        print("BEST FEATURE COMBINATION BY GROUP")
        print("="*70)

        best_attr = attr_means['mean'].idxmax()
        best_axis = axis_means['mean'].idxmax()
        best_freq = freq_means['mean'].idxmax()
        best_cycle = cycle_means['mean'].idxmax()

        print(f"Best Attribute: {best_attr}")
        print(f"Best Axis: {best_axis}")
        print(f"Best Frequency: {best_freq}")
        print(f"Best Cycle Region: {best_cycle}")

        # Find the actual feature that matches this combination
        best_match = df[
            (df['attribute'] == best_attr) &
            (df['axis'] == best_axis) &
            (df['frequency'] == best_freq) &
            (df['cycle_region'] == best_cycle)
        ]

        if not best_match.empty:
            print(f"\nBest Feature: {best_match.iloc[0]['feature']}")
            print(f"Score: {best_match.iloc[0]['discriminative_score']:.4f}")

    def plot_results(self):
        """
        Create comprehensive plots of feature selection results.
        """
        if 'lmm' not in self.results:
            print("Run evaluate_all_features() first")
            return

        try:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))

            df = self.results['lmm']

            # 1. Top LMM features
            ax = axes[0, 0]
            top_lmm = df.head(10)
            colors = ['red' if x < 0.05 else 'gray' for x in top_lmm['class_b_pvalue']]
            ax.barh(top_lmm['feature'], top_lmm['discriminative_score'], color=colors)
            ax.axvline(x=-np.log10(0.05), color='black', linestyle='--', label='p=0.05')
            ax.set_xlabel('Discriminative Score (-log10(p-value))')
            ax.set_title('Top 10 Features (LMM)')
            ax.invert_yaxis()
            ax.legend()

            # 2. P-values by feature
            ax = axes[0, 1]
            top_20 = df.head(20)
            x = np.arange(len(top_20))
            width = 0.35
            ax.bar(x - width/2, top_20['class_b_pvalue'], width, label='Class B', color='blue', alpha=0.7)
            ax.bar(x + width/2, top_20['class_c_pvalue'], width, label='Class C', color='red', alpha=0.7)
            ax.axhline(y=0.05, color='black', linestyle='--', label='p=0.05')
            ax.set_xlabel('Feature')
            ax.set_ylabel('p-value')
            ax.set_title('Class Difference p-values')
            ax.set_xticks(x)
            ax.set_xticklabels(top_20['feature'], rotation=45, ha='right')
            ax.legend()

            # 3. Feature groups comparison
            ax = axes[0, 2]
            group_colors = {'gain': 'blue', 'speed': 'green', 'velocity': 'red'}
            for attr in df['attribute'].unique():
                attr_data = df[df['attribute'] == attr]
                ax.scatter(attr_data['axis'] + '_' + attr_data['cycle_region'],
                          attr_data['discriminative_score'],
                          label=attr, color=group_colors.get(attr, 'gray'), alpha=0.7)
            ax.set_xlabel('Axis_CycleRegion')
            ax.set_ylabel('Discriminative Score')
            ax.set_title('Score by Attribute, Axis, and Cycle Region')
            ax.legend()
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

            # 4. Random Forest importance
            ax = axes[1, 0]
            if 'rf_agg_importance' in self.results:
                top_rf = self.results['rf_agg_importance'].head(10)
                ax.barh(top_rf.index, top_rf.values, color='green', alpha=0.7)
                ax.set_xlabel('Importance')
                ax.set_title('Top 10 Features (Random Forest)')
                ax.invert_yaxis()
            else:
                ax.text(0.5, 0.5, 'RF analysis not run', ha='center', va='center')

            # 5. Heatmap of feature performance
            ax = axes[1, 1]
            pivot = df.pivot_table(
                values='discriminative_score',
                index='attribute',
                columns='axis',
                aggfunc='mean'
            )
            sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
            ax.set_title('Score by Attribute × Axis')

            # 6. Summary statistics
            ax = axes[1, 2]
            ax.axis('off')

            # Create summary text
            summary_text = "FEATURE SELECTION SUMMARY\n"
            summary_text += "="*30 + "\n\n"

            # Best feature
            if not df.empty:
                best = df.iloc[0]
                summary_text += f"Best Feature:\n"
                summary_text += f"  {best['feature']}\n"
                summary_text += f"  Score: {best['discriminative_score']:.4f}\n\n"

            # Best group
            if 'group_analysis' in self.results:
                ga = self.results['group_analysis']
                summary_text += f"Best Attribute: {ga['attribute']['mean'].idxmax()}\n"
                summary_text += f"Best Axis: {ga['axis']['mean'].idxmax()}\n"
                summary_text += f"Best Frequency: {ga['frequency']['mean'].idxmax()}\n"
                summary_text += f"Best Cycle Region: {ga['cycle_region']['mean'].idxmax()}\n"

            ax.text(0.1, 0.5, summary_text, transform=ax.transAxes,
                   fontsize=10, verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"Error plotting: {e}")

    def get_recommendations(self, top_n=5):
        """
        Generate comprehensive recommendations.
        """
        print("\n" + "="*70)
        print("FEATURE SELECTION RECOMMENDATIONS")
        print("="*70)

        recommendations = {}

        # LMM recommendations
        if 'lmm' in self.results:
            top_lmm = self.results['lmm'].head(top_n)
            recommendations['lmm'] = {
                'features': top_lmm['feature'].tolist(),
                'scores': top_lmm['discriminative_score'].tolist()
            }

            print(f"\n✅ Top {top_n} Features (LMM):")
            for i, row in top_lmm.iterrows():
                print(f"  {i+1}. {row['feature']}")
                print(f"     Score: {row['discriminative_score']:.4f}")
                print(f"     Class B p-value: {row['class_b_pvalue']:.4f}")
                print(f"     Class C p-value: {row['class_c_pvalue']:.4f}")

        # RF recommendations
        if 'rf_agg_importance' in self.results:
            top_rf = self.results['rf_agg_importance'].head(top_n)
            recommendations['rf'] = {
                'features': top_rf.index.tolist(),
                'scores': top_rf.values.tolist()
            }

            print(f"\n✅ Top {top_n} Features (Random Forest):")
            for i, (feature, score) in enumerate(top_rf.items()):
                print(f"  {i+1}. {feature}: {score:.4f}")

        # Consensus (features in both top lists)
        if 'lmm' in recommendations and 'rf' in recommendations:
            lmm_set = set(recommendations['lmm']['features'])
            rf_set = set(recommendations['rf']['features'])
            consensus = lmm_set & rf_set

            if consensus:
                print(f"\n🌟 Consensus Features (appear in both methods):")
                for feature in consensus:
                    print(f"  ✓ {feature}")
            else:
                print(f"\n⚠️ No consensus between methods.")
                print(f"   LMM recommends: {list(lmm_set)[:5]}")
                print(f"   RF recommends: {list(rf_set)[:5]}")

        # Group recommendations
        if 'group_analysis' in self.results:
            ga = self.results['group_analysis']

            print(f"\n📊 Best Feature Groups:")
            print(f"  Attribute: {ga['attribute']['mean'].idxmax()} "
                  f"(score: {ga['attribute']['mean'].max():.3f})")
            print(f"  Axis: {ga['axis']['mean'].idxmax()} "
                  f"(score: {ga['axis']['mean'].max():.3f})")
            print(f"  Frequency: {ga['frequency']['mean'].idxmax()} "
                  f"(score: {ga['frequency']['mean'].max():.3f})")
            print(f"  Cycle Region: {ga['cycle_region']['mean'].idxmax()} "
                  f"(score: {ga['cycle_region']['mean'].max():.3f})")

        print("\n" + "="*70)
        print("RECOMMENDED FEATURE SET FOR CLASSIFICATION")
        print("="*70)

        # Build recommended feature set
        recommended_features = []

        if 'lmm' in recommendations:
            # Use top LMM features as primary recommendation
            recommended_features = recommendations['lmm']['features'][:top_n]

        if 'rf' in recommendations and len(recommended_features) < top_n:
            # Add RF features that aren't already included
            for feature in recommendations['rf']['features']:
                if feature not in recommended_features:
                    recommended_features.append(feature)
                    if len(recommended_features) >= top_n * 2:
                        break

        print(f"\nRecommended Features for your compressed dataset:")
        for i, feature in enumerate(recommended_features[:top_n * 2]):
            print(f"  {i+1}. {feature}")

        return recommendations

In [ ]:
# Set your data directory
DATA_DIR = "./"

# Initialize pipeline
pipeline = FeatureSelectionPipeline(DATA_DIR, verbose=True)

# 1. Load all files
pipeline.load_all_files()

In [ ]:
# 1a. Save results
output_dir = Path("feature_selection_results")
output_dir.mkdir(exist_ok=True)

# 2. Evaluate all features with LMM
lmm_results = pipeline.evaluate_all_features()
# Save LMM results
if 'lmm' in pipeline.results:
    pipeline.results['lmm'].to_csv(output_dir / 'lmm_feature_scores.csv', index=False)

In [ ]:
# 3. Random Forest importance (optional - may be slow for large data)
rf_results = pipeline.random_forest_importance()
# Save RF importance
if 'rf_agg_importance' in pipeline.results:
    pipeline.results['rf_agg_importance'].to_csv(output_dir / 'rf_feature_importance.csv',
                                                  index=True, header=['importance'])


FEATURE IMPORTANCE USING RANDOM FOREST
Number of features: 54
Number of statistics per feature: 8
Total feature dimensions: 432

Checking class sizes:
  Class A: 85 subjects
  Class B: 72 subjects
  Class C: 62 subjects
Total subjects: 219


IndexError: list index out of range

In [ ]:
# 4. Analyze by feature groups
pipeline.analyze_by_feature_groups()
# Save group analysis
if 'group_analysis' in pipeline.results:
    for group, df in pipeline.results['group_analysis'].items():
        df.to_csv(output_dir / f'group_analysis_{group}.csv')

In [ ]:
# 5. Plot results
pipeline.plot_results()

# 6. Get recommendations
recommendations = pipeline.get_recommendations(top_n=5)

In [ ]:
pipeline.features_data

Buffered data was truncated after reaching the output size limit.